# Data Quality & Cleaning

This notebook focuses on validating and improving the quality of the FinSight source data before further analysis and modeling.

## Objectives

- identify duplicate records and validate key uniqueness,
- analyze missing values,
- correct inconsistent data types,
- validate relationships between datasets,
- detect invalid or anomalous values,
- build reusable data validation checks,
- generate a data quality summary,
- save cleaned datasets to `data/processed/`.

> Raw data in `data/raw/` remains unchanged throughout the process.

In [1]:
import json
from pathlib import Path

import pandas as pd

from finsight.validation import (
    check_foreign_keys,
    check_duplicates,
    check_key_duplicates,
    check_missing_values,
    check_amounts,
)

project_root = Path.cwd().parent

print("FinSight environment ready")

FinSight environment ready


## Load Source Data

In [2]:
users = pd.read_csv("../data/raw/users_data.csv")
transactions = pd.read_csv("../data/raw/transactions_data.csv")
cards = pd.read_csv("../data/raw/cards_data.csv")
with open("../data/raw/mcc_codes.json", "r") as file:
    mcc_dict = json.load(file)
codes = pd.DataFrame(
    mcc_dict.items(),
    columns=["mcc", "description"]
)
with open("../data/raw/train_fraud_labels.json", "r") as file:
    fraud_dict = json.load(file)
fraud_target = fraud_dict["target"]
fraud = pd.DataFrame(
    fraud_target.items(),
    columns=["transaction_id", "is_fraud"]
)

print("Loading raw files ready")

Loading raw files ready


## Duplicate Validation

### Users

In [3]:
users.duplicated().sum()

np.int64(0)

**Result:** No fully duplicated rows were found in `users`.

In [4]:
users["id"].duplicated().sum()

np.int64(0)

**Result:** No duplicate `id` values were found. Each user has a unique identifier.

### Cards

In [5]:
cards.duplicated().sum()

np.int64(0)

**Result:** No fully duplicated rows were found in `cards`.

#### Primary Key Duplicate Check

Check whether `id` contains duplicate values, as each card should have a unique identifier.

In [6]:
cards["id"].duplicated().sum()

np.int64(0)

**Result:** No duplicate `id` values were found. Each card has a unique identifier.

### Transactions

In [7]:
transactions.duplicated().sum()

np.int64(0)

**Result:** No fully duplicated rows were found in `transactions`.

In [ ]:
transactions["id"].duplicated().sum()

**Result:** No duplicate `id` values were found. Each transaction has a unique identifier.

### MCC Codes

In [ ]:
codes.duplicated().sum()

**Result:** No fully duplicated rows were found in `codes`.

In [ ]:
codes["mcc"].duplicated().sum()

**Result:** No duplicate `mcc` values were found. Each code has a unique identifier.

### Fraud Labels

In [ ]:
fraud.duplicated().sum()

**Result:** No fully duplicated rows were found in `fraud`.

In [ ]:
fraud["transaction_id"].duplicated().sum()

**Result:** No duplicate `transaction_id` values were found. Each transaction has at most one fraud label.

## Missing Values Validation

### Users

In [ ]:
users.isna().sum()

**Result:** No missing values were found in `users`.

### Cards

In [ ]:
cards.isna().sum()

**Result:** No missing values were found in `cards`.

### Transactions

In [ ]:
transactions.isna().sum()

#### Missing Values Percentage

Calculate the percentage of missing values in each column of `transactions`.

In [ ]:
percentage = round(((transactions.isna().sum())/len(transactions)) * 100, 2)
percentage

#### Transaction Errors

Inspect the values present in `errors` to determine whether missing values represent transactions without an error.

In [ ]:
transactions["errors"].value_counts()

#### Transaction Errors — Missing Values

Check whether missing values in `errors` represent transactions with no recorded error by including missing values in the frequency count.

In [ ]:
transactions["errors"].value_counts(dropna = False)

**Result:** The `errors` column contains 98.41% missing values. These values represent transactions with no recorded error rather than a data quality issue. Non-null values contain specific transaction error types.

#### Missing `merchant_state` Investigation

Inspect transactions with missing `merchant_state` values to determine whether the missingness is expected or indicates a data quality issue.

In [ ]:
transactions[transactions["merchant_state"].isna()].head(5)

#### Missing `merchant_state` by Transaction Type

Check whether missing `merchant_state` values occur exclusively in online transactions.

In [ ]:
transactions[
    transactions["merchant_state"].isna()
]["use_chip"].unique()

#### Chip Transactions with Missing `merchant_state`

Inspect chip transactions where `merchant_state` is missing to determine whether there is another pattern explaining the missing location data.

In [ ]:
transactions[
    (transactions["merchant_state"].isna()) &
    (transactions["use_chip"] == "Chip Transaction")
].head()

#### Missing `merchant_state` by Merchant Location

Check whether transactions with missing `merchant_state` are associated with `merchant_city = ONLINE`.

In [ ]:
transactions[
    transactions["merchant_state"].isna()
]["merchant_city"].value_counts()

**Result:** All 1,563,700 transactions with missing `merchant_state` have `merchant_city = ONLINE`. Therefore, the missing values are expected for online transactions and do not indicate a data quality issue.

#### Missing `zip` Investigation

Inspect transactions with missing `zip` values to determine whether the missingness is expected or indicates a data quality issue.

In [ ]:
transactions[
    transactions["zip"].isna()
].head()

#### Missing `zip` by Merchant Location

Check which merchant locations are associated with missing `zip` values.

In [ ]:
transactions[
    transactions["zip"].isna()
]["merchant_city"].value_counts()

#### Missing `zip` by Merchant State

Inspect `merchant_state` values for transactions with missing `zip` to determine whether the additional missing ZIP codes are associated with international transactions.

In [ ]:
transactions[
    transactions["zip"].isna()
]["merchant_state"].value_counts(dropna=False)

**Result:** Missing `zip` values are primarily associated with online and international transactions. Online transactions have no physical merchant location, while international transactions may not use the ZIP code format represented by this field. Therefore, these missing values appear to be structurally expected rather than a data quality issue.

### MCC Codes

In [ ]:
codes.isna().sum()

**Result:** No missing values were found in `codes`.

### Fraud Labels

In [ ]:
fraud.isna().sum()

**Result:** No missing values were found in `fraud`.

## Data Type Validation

Review column data types and convert fields where necessary to ensure dates, numeric values, and categorical data are represented correctly.

### Users

Inspect the current data types in `users`.

In [ ]:
users.dtypes

#### Monetary Columns

Inspect monetary columns currently stored as strings before converting them to numeric data types.

In [ ]:
users[["per_capita_income", "yearly_income", "total_debt"]].head()

In [ ]:
users["per_capita_income"] = (
    users["per_capita_income"]
    .str.replace("$", "", regex=False)
)
users["per_capita_income"].head()


In [ ]:
users["per_capita_income"] = pd.to_numeric(
    users["per_capita_income"]
)
users["per_capita_income"].dtype

In [ ]:
users["yearly_income"] = (
    users["yearly_income"]
    .str.replace("$", "", regex=False)
)
users["yearly_income"].head()

In [ ]:
users["yearly_income"] = pd.to_numeric(
    users["yearly_income"]
)
users["yearly_income"].dtype

In [ ]:
users["total_debt"] = (
    users["total_debt"]
    .str.replace("$", "", regex=False)
)
users["total_debt"].head()

In [ ]:
users["total_debt"] = pd.to_numeric(
    users["total_debt"]
)
users["total_debt"].dtype

**Result:** The monetary columns `per_capita_income`, `yearly_income`, and `total_debt` were converted from strings containing currency symbols to numeric (`int64`) values. The remaining columns already use appropriate data types.

### Cards

Inspect the current data types in `cards` and identify columns that require conversion.

In [ ]:
cards.dtypes

#### Columns Requiring Inspection

Inspect columns that may require data type conversion before applying any transformations.

In [ ]:
cards[
    ["expires", "has_chip", "credit_limit", "acct_open_date", "card_on_dark_web"]
].head()

#### Convert `expires`

Convert the card expiration month and year from string format to datetime.

In [ ]:
cards["expires"] = pd.to_datetime(
    cards["expires"],
    format="%m/%Y"
)
cards["expires"].head()

#### Convert `has_chip`

Convert `has_chip` from `YES` / `NO` string values to boolean values for clearer and more efficient analysis. 

In [ ]:
cards["has_chip"].value_counts(dropna=False)

In [ ]:
cards["has_chip"] = cards["has_chip"].map({
    "YES": True,
    "NO": False                                      
})
cards["has_chip"].head()

In [ ]:
cards["has_chip"].dtype

#### Convert `credit_limit`

Remove the currency symbol from `credit_limit` and convert the column from string to a numeric data type.

In [ ]:
cards["credit_limit"] = (
    cards["credit_limit"]
    .str.replace("$", "", regex=False)
)
cards["credit_limit"].head()

In [ ]:
cards["credit_limit"] = pd.to_numeric(
    cards["credit_limit"]
)
cards["credit_limit"].dtype

#### Convert `acct_open_date`

Convert `acct_open_date` from month/year string format to datetime.

In [ ]:
cards["acct_open_date"] = pd.to_datetime(
    cards["acct_open_date"],
    format="%m/%Y"
)
cards["acct_open_date"].head()

#### Convert `card_on_dark_web`

Convert `card_on_dark_web` from `Yes` / `No` string values to boolean values for clearer analysis.

In [ ]:
cards["card_on_dark_web"].value_counts(dropna=False)

**Result:** All 6,146 cards have `card_on_dark_web = No`. The column is constant and therefore carries no analytical or predictive information. It is retained for completeness but excluded from modeling.

In [ ]:
cards["card_on_dark_web"] = cards["card_on_dark_web"].map({
    "Yes": True,
    "No": False
})

cards["card_on_dark_web"].dtype

### Transactions

Inspect the current data types in `transactions` and identify columns that require conversion.

In [ ]:
transactions.dtypes

In [ ]:
transactions[["date", "amount"]].head()

#### Convert `date`

Convert the transaction timestamp from string format to datetime for time-based analysis.

In [ ]:
transactions["date"] = pd.to_datetime(
    transactions["date"],
    format="%Y-%m-%d %H:%M:%S"
)

transactions["date"].dtype

#### Convert `amount`

Remove the currency symbol from `amount` and convert the column from string to a numeric data type while preserving decimal values.

In [ ]:
transactions["amount"] = (
    transactions["amount"]
    .str.replace("$", "", regex = False)
)
transactions["amount"].head()

In [ ]:
transactions["amount"] = pd.to_numeric(
    transactions["amount"]
)
transactions["amount"].dtype

### MCC Codes

Inspect the current data types in `codes` and identify whether any columns require conversion.

In [ ]:
codes.dtypes

In [ ]:
codes["mcc"] = pd.to_numeric(
    codes["mcc"]
)
codes["mcc"].dtype

### Fraud Labels

Inspect the current data types in `fraud` and identify whether any columns require conversion.

In [ ]:
fraud.dtypes

In [ ]:
fraud["transaction_id"] = pd.to_numeric(
    fraud["transaction_id"]
)
fraud["transaction_id"].dtype

In [ ]:
fraud["is_fraud"].value_counts()

In [ ]:
fraud["is_fraud"] = fraud["is_fraud"].map({
    "Yes": True,
    "No": False                                      
})
fraud["is_fraud"].head()

## Relationship Integrity Validation

Validate key relationships between datasets to identify transactions referencing non-existent users, cards, or MCC codes.

### Transactions → Users

Check whether every `client_id` in `transactions` corresponds to an existing `id` in `users`.

In [ ]:
check_foreign_keys(transactions, users, "client_id", "id")

**Result:** No orphan `client_id` records were found. Every transaction references an existing user.

### Transactions → Cards

Check whether every `card_id` in `transactions` corresponds to an existing `id` in `cards`.

In [ ]:
check_foreign_keys(transactions, cards, "card_id", "id")

**Result:** No orphan `card_id` records were found. Every transaction references an existing card.

### Transactions → MCC Codes

Check whether every `mcc` in `transactions` corresponds to an existing `mcc` in `codes`.

In [ ]:
check_foreign_keys(transactions, codes, "mcc", "mcc")

**Result:** No orphan `mcc` records were found. Every transaction references an existing code.

### Fraud Labels → Transactions

Check whether every `transaction_id` in `fraud` corresponds to an existing `id` in `transactions`.

In [ ]:
check_foreign_keys(fraud, transactions, "transaction_id", "id")

**Result:** No orphan `transaction_id` records were found. Every fraud label references an existing transaction.

### Cards → Users

Check whether every `client_id` in `cards` corresponds to an existing `id` in `users`.

In [ ]:
check_foreign_keys(cards, users, "client_id", "id")

**Result:** No orphan `client_id` records were found. Every card belongs to an existing user.

## Reusable Data Validation

The manual checks above were refactored into reusable functions in `src/finsight/validation.py`. This section verifies that the reusable functions reproduce the results of the manual checks, using the `users` dataset as an example.

### Duplicate Validation - Users

Validate the `users` dataset using the reusable duplicate-check function.

In [ ]:
check_duplicates(users)

### Primary Key Validation - Users

Validate that `id` remains unique in the `users` dataset.

In [ ]:
check_key_duplicates(users, "id")

### Missing Values Validation - Users

Validate missing values in each column of the cleaned `users` dataset.

In [ ]:
check_missing_values(users)

**Result:** The reusable functions return the same results as the manual checks: no duplicated rows, no duplicated primary keys and no missing values in `users`.

### Transaction Amount Validation

Validate transaction amounts by identifying negative and zero-value transactions.

In [ ]:
check_amounts(transactions, "amount")

**Result:** 660,049 transactions (4.96%) have negative amounts and 10,639 (0.08%) have zero amounts.

Negative amounts are retained, as they likely represent refunds or reversals rather than data errors. They are excluded from gross transaction value and included in net transaction value in the SQL analytics layer.

## Data Quality Report

Generate a consolidated summary of data quality metrics across all datasets using the reusable validation functions.

In [ ]:
quality_report = pd.DataFrame([
    {
        "dataset": "users",
        "rows": len(users),
        "duplicates": check_duplicates(users),
        "key_duplicates": check_key_duplicates(users, "id"),
        "missing_values": check_missing_values(users).sum()
    },
    {
        "dataset": "cards",
        "rows": len(cards),
        "duplicates": check_duplicates(cards),
        "key_duplicates": check_key_duplicates(cards, "id"),
        "missing_values": check_missing_values(cards).sum()
    },
    {
        "dataset": "transactions",
        "rows": len(transactions),
        "duplicates": check_duplicates(transactions),
        "key_duplicates": check_key_duplicates(transactions, "id"),
        "missing_values": check_missing_values(transactions).sum()
    },
    {
        "dataset": "codes",
        "rows": len(codes),
        "duplicates": check_duplicates(codes),
        "key_duplicates": check_key_duplicates(codes, "mcc"),
        "missing_values": check_missing_values(codes).sum()
    },
    {
        "dataset": "fraud",
        "rows": len(fraud),
        "duplicates": check_duplicates(fraud),
        "key_duplicates": check_key_duplicates(fraud, "transaction_id"),
        "missing_values": check_missing_values(fraud).sum()
    }
])

quality_report

**Result:** None of the five datasets contain duplicated rows or duplicated primary keys. Missing values occur only in `transactions` and are analyzed in detail below.

### Missing Values Breakdown - Transactions

Missing values are concentrated in the `transactions` dataset. Display the number and percentage of missing values for each affected column.

In [ ]:
transactions_missing = pd.DataFrame({
    "missing_values": transactions.isna().sum(),
    "missing_percentage": (
        transactions.isna().mean() * 100
    ).round(2)
})

transactions_missing = transactions_missing[
    transactions_missing["missing_values"] > 0
]

transactions_missing

**Result:** Missing values are limited to `merchant_state`, `zip`, and `errors`. 
The missing location fields are primarily associated with online and international 
transactions, where these attributes may not be applicable. Missing values in 
`errors` represent transactions for which no error was recorded.

These missing values are therefore treated as structurally expected and are 
retained rather than imputed or removed.

## Save Processed Data

Save the cleaned and validated datasets in Parquet format to `data/processed/`.

Parquet is used to preserve data types, reduce storage size, and improve read performance. The original files in `data/raw/` remain unchanged.

In [ ]:
processed_path = project_root / "data" / "processed"

processed_path.mkdir(parents=True, exist_ok=True)

### Sensitive Data Removal

`card_number` and `cvv` are removed before saving. They have no analytical or predictive value, and storing full card numbers with verification codes would violate data minimization principles (e.g. PCI DSS) in a real payment environment.

In [ ]:
cards = cards.drop(columns=["card_number", "cvv"])
cards.columns.tolist()

In [ ]:
users.to_parquet(
    processed_path / "users.parquet",
    index=False
)

cards.to_parquet(
    processed_path / "cards.parquet",
    index=False
)

transactions.to_parquet(
    processed_path / "transactions.parquet",
    index=False
)

codes.to_parquet(
    processed_path / "mcc_codes.parquet",
    index=False
)

fraud.to_parquet(
    processed_path / "fraud_labels.parquet",
    index=False
)

print("Processed datasets saved successfully.")

### Processed Data Validation

Verify that all processed Parquet files were created successfully and can be read correctly.

In [ ]:
for file in processed_path.glob("*.parquet"):
    print(file.name)

In [ ]:
cards_test = pd.read_parquet(
    processed_path / "cards.parquet"
)

cards_test.dtypes

**Result:** All processed datasets were successfully saved in Parquet format. A validation read confirmed that the transformed data types were preserved correctly.

## Final Summary

The data quality stage successfully validated and prepared all five source datasets for downstream analysis.

Key outcomes:
- no fully duplicated rows or duplicated primary keys were identified,
- structurally expected missing values were investigated and retained,
- monetary, datetime, boolean, and identifier fields were converted to appropriate data types,
- all key relationships were validated with no orphan records,
- reusable validation functions were implemented in `src/finsight/validation.py`,
- cleaned datasets were stored in Parquet format with transformed data types preserved.
- sensitive card data (`card_number`, `cvv`) was removed before saving,

The processed datasets are now ready for database loading, analytical querying, and further exploratory analysis.